In [ ]:
import random
import glob
import json
import cv2
import numpy as np
import os

from ultralytics import YOLO
# 初期設定
yolo_model = YOLO('models/tableware_best_ncnn_model', task='detect', verbose=False)

In [ ]:
# テスト用基準データ作成

# 軟菜食のみ，固形食のみ，両方混在の3パターンを指定
PLATE_TYPE_MODE = "both" # "nansai" or "kokei" or "both"

# 深度層分散計算関数
def calc_depth_var(depth):
    # 外周外マスク
    h, w = depth.shape
    center = (w // 2, h // 2)
    radius = min(h, w) // 2
    mask = np.zeros((h, w), dtype=np.uint8)
    mask = cv2.circle(mask, center, radius, 1, -1)
    depth_masked = depth * mask
    # 欠損0を除く
    data = depth_masked.flatten()
    data = np.delete(data, data==0)
    BIN_COUNT = 50
    DATA_MIN = np.min(data)
    DATA_MAX = np.max(data)
    bins = np.linspace(DATA_MIN, DATA_MAX, BIN_COUNT + 1)

    hist, _ = np.histogram(data, bins=bins)
    nums = [d/data.size for d in hist.tolist()]
    return np.var(nums)


# 食器の種類を読み込み
with open("config.json") as f:
    config = json.load(f)

# メニューの種類を読み込み
tray_ids = [d["id"] for d in config["shokushu"]]

# 食器の種類を読み込み
plate_names = list(config.keys())
plate_names.remove('shokushu')

for tray_id in tray_ids:
    # 空の食器を基準とする場合（10割は00を10に変更）
    plate_ids = "00" * len(plate_names)
    plate_images = glob.glob(f"captured_images/captured_images_{tray_id}/{tray_id}_{plate_ids}*.png")
    if not plate_images:
        continue

    # 食器登録（test_dataディレクトリ内）
    image_path = random.choice(plate_images)
    os.makedirs(f"test_data/{tray_id}", exist_ok=True)

    results = yolo_model(image_path, verbose=False)
    result = results[0]
    boxes = result.boxes.xyxy.cpu().numpy()

    depth_var_list = []
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        dish_image = cv2.imread(image_path)[y1:y2, x1:x2]
        cv2.imwrite(f"test_data/{tray_id}/plate_{i}.png", dish_image)
        # 深度層の分散による固形軟菜の判定
        dish_depth = np.load(image_path.replace(".png", ".npy"))[y1:y2, x1:x2]
        depth_var = calc_depth_var(dish_depth)
        depth_var_list.append(depth_var)

    if PLATE_TYPE_MODE == "nansai":
        # 軟菜食のみ
        binary = [True] * len(depth_var_list)
    elif PLATE_TYPE_MODE == "kokei":
        # 固形食のみ
        binary = [False] * len(depth_var_list)
    elif PLATE_TYPE_MODE == "both":
        # 固形軟菜判定のしきい値を決定
        depth_var_list = np.array(depth_var_list)
        sorted_idx = np.argsort(depth_var_list)
        sorted_arr = depth_var_list[sorted_idx]
        diff = np.diff(sorted_arr)
        max_gap_idx = np.argmax(diff)
        threshold = (sorted_arr[max_gap_idx] + sorted_arr[max_gap_idx + 1]) / 2
        binary = np.array(depth_var_list) > threshold
        binary = binary.astype(bool).tolist()

    plate_setting = [{"plate_name": plate_names[i], "plate_number": i, "nansai": nansai_flg} for i, nansai_flg in enumerate(binary)]
    json.dump(plate_setting, open(f"test_data/{tray_id}/plate_setting.json", "w"), indent=4)
    print(f"基準データを作成しました: test_data/{tray_id}/")
    print(f"plate_setting.jsonを編集し、plate_nameを設定してください。")

In [ ]:
import random
import glob
import json
import cv2
import numpy as np
import os
import shutil

from ultralytics import YOLO
# 初期設定
yolo_model = YOLO('models/tableware_best_ncnn_model', task='detect', verbose=False)


# 食器の切り出し


# YOLO前回実行時のキャッシュを利用するかどうか（Trueにすると高速化されるが、前回と同じ順番で画像が処理されることが前提）
USE_YOLO_CACHE = False
if USE_YOLO_CACHE and os.path.exists("boxes_cache.npy"):
    boxes_cache = np.load("boxes_cache.npy", allow_pickle=True)
    if len(boxes_cache) != len(glob.glob(f"captured_images/*/*.png")):
        print("警告: キャッシュの食器数が現在の画像数と異なります。キャッシュを使用せずに再実行します。")
        boxes_cache = []
else:
    boxes_cache = []


all_plate_N = 0
detect_plate_N = 0

plate_images = sorted(glob.glob(f"captured_images/*/*.png"))
error_count = 0


import os
import glob
import cv2  # 画像描画と保存のために追加
import numpy as np

for image_idx, plate_image in enumerate(plate_images):
    plate_name = os.path.splitext(os.path.basename(plate_image))[0]
    dir_path = f"plate_images_tmp/{plate_name}"
    os.makedirs(dir_path, exist_ok=True)
    
    if USE_YOLO_CACHE:
        boxes = boxes_cache[image_idx]
        if np.any(np.isnan(boxes)):
            print(f"警告: {plate_name}の食器数が基準データと異なります（キャッシュ使用）。")
            error_count += 1
            continue
    else:
        results = yolo_model(plate_image, verbose=False)
        boxes = results[0].boxes
        sorted_boxes = boxes[boxes.conf.argsort(descending=True)]
        N = 5
        boxes = sorted_boxes[:N].xyxy.cpu().numpy()
        
        # 食器数照合用の基準データを取得
        plate_id = plate_name.split("_")[0]
        gt_images = glob.glob(f"test_data/{plate_id}/*.png")
        gt_count = len(gt_images)
        
        all_plate_N += gt_count
        detect_plate_N += len(boxes)
        
        # 食器数が異なる場合の処理
        if len(boxes) != gt_count:
            error_count += 1
            print(f"警告: {plate_name}の食器数が基準データと異なります。{len(boxes)} vs {gt_count}")
            
            # --- 【追加】バウンディングボックスを描画して保存 ---
            img = cv2.imread(plate_image)
            if img is not None:
                for box in boxes:
                    # 座標を取得 (x1, y1, x2, y2)
                    x1, y1, x2, y2 = map(int, box[:4])
                    # 赤色の枠線を描画 (B=0, G=0, R=255), 線の太さ=2
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                
                # エラー画像の保存パス (例: plate_images_tmp/xxx/error.png)
                save_path = os.path.join(dir_path, f"{plate_name}_error.png")
                cv2.imwrite(save_path, img)
            # ------------------------------------------------
            
            # 元のロジック: ダミーのボックスを追加してキャッシュに保存
            while True:
                boxes = np.append(boxes, np.full((1, 4), np.nan), axis=0)
                if len(boxes) == gt_count:
                    break
            
            boxes_cache.append(boxes)
            continue
        else:
            boxes_cache.append(boxes)



    H, W = cv2.imread(plate_image).shape[:2]
    # 食器を切り出して保存（画像，深度）
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        dish_image = cv2.imread(plate_image)[y1:y2, x1:x2]
        dish_depth = np.load(plate_image.replace(".png", ".npy"))[y1:y2, x1:x2]

        # 回転補正
        def rotate_by_region(img, depth, cx, cy, W, H):
            # 領域判定
            if cx < W/2 and cy < H/2:
                k = 0  # 左上
            elif cx >= W/2 and cy < H/2:
                k = 1  # 右上
            elif cx >= W/2 and cy >= H/2:
                k = 2  # 右下
            else:
                k = 3  # 左下
            
            # numpy回転（90度単位）
            img_rot = np.rot90(img, k=k)
            depth_rot = np.rot90(depth, k=k)
            
            return img_rot, depth_rot, k

        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        dish_image, dish_depth, region = rotate_by_region(
            dish_image, dish_depth, cx, cy, W, H
        )
        
        os.makedirs(f"plate_images_tmp/{plate_name}", exist_ok=True)
        
        cv2.imwrite(f"plate_images_tmp/{plate_name}/plate_{i}_tmp.png", dish_image)
        np.save(f"plate_images_tmp/{plate_name}/plate_{i}_tmp.npy", dish_depth)



if not USE_YOLO_CACHE:
    np.save("boxes_cache.npy", np.array(boxes_cache))

print(f"切り出し完了。プレート総数: {len(plate_images)}、エラー数: {error_count}、エラー率: {error_count / len(plate_images) * 100:.2f}%")
print(f"総食器数: {all_plate_N}、検出食器数: {detect_plate_N}")

In [ ]:
#複合特徴量マッチング

# ハンガリアンマッチングで食器の種別を判定
import shutil
import glob
import os
import cv2
import numpy as np
import json
from scipy.optimize import linear_sum_assignment


def calc_feature(img):
    h, w = img.shape[:2]

    # 食器サイズ
    area = h * w

    # 白色率
    white_mask = (
        (img[:, :, 0] > 180)
        & (img[:, :, 1] > 180)
        & (img[:, :, 2] > 180)
    )
    white_ratio = white_mask.mean()

    # 平均彩度
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mean_s = hsv[:, :, 1].mean()

    # エッジ量
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edge = cv2.Canny(gray, 50, 150)
    edge_ratio = np.count_nonzero(edge) / edge.size

    return np.array(
        [
            area,
            white_ratio,
            mean_s,
            edge_ratio,
        ],
        dtype=np.float32,
    )


def get_matched_index(features1, features2):
    features1 = np.array(features1, dtype=np.float32)
    features2 = np.array(features2, dtype=np.float32)

    # 特徴量を標準化
    all_features = np.vstack([features1, features2])

    mean = all_features.mean(axis=0)
    std = all_features.std(axis=0)

    std[std < 1e-6] = 1.0

    features1 = (features1 - mean) / std
    features2 = (features2 - mean) / std

    # 特徴量の重み
    weights = np.array(
        [
            1.0,  # 面積
            0,  # 白色率
            1.0,  # 彩度
            0,  # エッジ量
        ],
        dtype=np.float32,
    )

    cost_matrix = np.zeros(
        (len(features2), len(features1)),
        dtype=np.float32,
    )

    for i, f2 in enumerate(features2):
        for j, f1 in enumerate(features1):
            diff = (f1 - f2) * weights
            cost_matrix[i, j] = np.linalg.norm(diff)

    _, col_ind = linear_sum_assignment(cost_matrix)

    return col_ind.tolist()


plate_names = [
    os.path.basename(d)
    for d in glob.glob("plate_images_tmp/*")
]

for plate_name in plate_names:

    # プレート特徴量
    features = []

    plate_images_tmp = sorted(
        glob.glob(
            f"plate_images_tmp/{plate_name}/*_tmp.png"
        )
    )
    if len(plate_images_tmp) == 0:
        print(f"警告: {plate_name}の切り出し画像が存在しません。")
        continue

    for plate_image_tmp in plate_images_tmp:
        img = cv2.imread(plate_image_tmp)

        if img is None:
            continue

        feature = calc_feature(img)
        features.append(feature)

    # 基準特徴量
    base_features = []

    plate_id = plate_name.split("_")[0]

    base_images = sorted(
        glob.glob(
            f"test_data/{plate_id}/*.png"
        )
    )

    for plate_image in base_images:
        img = cv2.imread(plate_image)

        if img is None:
            continue

        feature = calc_feature(img)
        base_features.append(feature)

    matched_indices = get_matched_index(
        base_features,
        features,
    )

    base_config = json.load(
        open(
            f"test_data/{plate_id}/plate_setting.json",
            encoding="utf-8",
        )
    )

    idx_to_plate_type = {
        d["plate_number"]: d["plate_name"]
        for d in base_config
    }

    for i, idx in enumerate(matched_indices):

        meal_type = idx_to_plate_type[idx]

        if meal_type == "shushoku":
            amount_idx = [0, 2]

        elif meal_type == "shusai":
            amount_idx = [2, 4]

        else:
            amount_idx = [4, 6]

        amount = plate_name.split("_")[1][
            amount_idx[0]:amount_idx[1]
        ]

        timestamp = "_".join(
            plate_name.split("_")[-2:]
        )

        os.makedirs(
            f"plate_images/{plate_id}/{meal_type}",
            exist_ok=True,
        )

        src_png = plate_images_tmp[i]
        dst_png = (
            f"plate_images/{plate_id}/{meal_type}/"
            f"plate_{amount}_{timestamp}.png"
        )

        src_npy = src_png.replace(".png", ".npy")
        dst_npy = dst_png.replace(".png", ".npy")

        os.rename(src_png, dst_png)

        if os.path.exists(src_npy):
            os.rename(src_npy, dst_npy)


# 一時ディレクトリ削除
from pathlib import Path

if os.path.exists("plate_images_tmp"):
    total_size = sum(
        f.stat().st_size
        for f in Path("plate_images_tmp").rglob("*")
        if f.is_file()
    )

    if total_size == 0:
        shutil.rmtree("plate_images_tmp")
        print("食器の切り出しと分類が完了しました。")